In [1]:
# =============================================================
# Cell 1 — Setup & A/B Testing Framework
# =============================================================
# We build a PromptVariant class and an A/B testing harness that
# runs multiple prompt versions against the same claims, measuring
# quality, token usage, and latency for each.
# =============================================================

import boto3
import json
import time
from datetime import datetime
from dataclasses import dataclass, field
from typing import Optional

bedrock = boto3.client('bedrock-runtime', region_name='us-east-1')
model_id = 'us.anthropic.claude-sonnet-4-5-20250929-v1:0'

# ----- Utility: Call Bedrock with timing -----
def ask(prompt, system=None, max_tokens=1024, temperature=0.0):
    """Call Bedrock and return response with metrics."""
    kwargs = {
        'modelId': model_id,
        'messages': [{'role': 'user', 'content': [{'text': prompt}]}],
        'inferenceConfig': {'maxTokens': max_tokens, 'temperature': temperature}
    }
    if system:
        kwargs['system'] = [{'text': system}]
    
    start = time.time()
    resp = bedrock.converse(**kwargs)
    latency = time.time() - start
    
    content = resp['output']['message'].get('content', [])
    text = content[0]['text'] if content and 'text' in content[0] else '[EMPTY RESPONSE]'
    
    return {
        'text':          text,
        'input_tokens':  resp['usage']['inputTokens'],
        'output_tokens': resp['usage']['outputTokens'],
        'latency':       round(latency, 2),
        'stop_reason':   resp.get('stopReason', 'unknown')
    }


# ----- Test Claims (same ones from our evaluation framework) -----
TEST_CLAIMS = [
    {
        'id': 'CLM-001',
        'type': 'auto',
        'description': 'Rear-end collision on Highway 401. Three vehicles involved. '
                       'Claimant reports whiplash and vehicle damage to rear bumper and trunk. '
                       'Police report filed. Other driver cited for following too closely. '
                       'Estimated vehicle repair: $8,500. Medical bills to date: $3,200.',
        'policy_limit': 100000
    },
    {
        'id': 'CLM-002',
        'type': 'homeowners',
        'description': 'Kitchen fire caused by unattended cooking. Fire department responded '
                       'and contained fire to kitchen area. Significant smoke damage throughout '
                       'first floor. Kitchen requires full renovation. Temporary housing needed '
                       'for estimated 6-8 weeks. Estimated damages: $45,000 structure, '
                       '$12,000 contents. No injuries reported.',
        'policy_limit': 300000
    },
    {
        'id': 'CLM-003',
        'type': 'auto',
        'description': 'Single-vehicle accident. Claimant hit a deer on rural road at dusk. '
                       'Front-end damage including hood, grille, headlights, and radiator. '
                       'Vehicle towed from scene. No injuries. Comprehensive coverage applies. '
                       'Estimated repair: $6,200. Vehicle value: $18,000.',
        'policy_limit': 50000
    }
]


# ----- Prompt Variants -----
# We test four versions of the same task, ranging from minimal to fully optimized.

@dataclass
class PromptVariant:
    """A named prompt configuration for A/B testing."""
    name: str
    system_prompt: Optional[str]
    user_template: str       # must contain {claim_text} placeholder
    description: str = ''    # what makes this variant different


# Variant A: Minimal — no system prompt, bare user prompt
VARIANT_A = PromptVariant(
    name='A_minimal',
    system_prompt=None,
    user_template='Analyze this insurance claim:\n\n{claim_text}',
    description='No system prompt. Minimal user instruction.'
)

# Variant B: Basic system prompt + structured user prompt
VARIANT_B = PromptVariant(
    name='B_structured',
    system_prompt=(
        'You are a P&C insurance claims analyst. '
        'Provide clear, structured claim assessments.'
    ),
    user_template=(
        'Analyze the following {claim_type} insurance claim and provide:\n'
        '1. Coverage determination\n'
        '2. Key risk factors\n'
        '3. Recommended next steps\n\n'
        'Claim: {claim_text}'
    ),
    description='Basic persona + structured output request.'
)

# Variant C: Full system prompt (persona + scope + guardrails) + CoT
VARIANT_C = PromptVariant(
    name='C_full_cot',
    system_prompt=(
        'You are Sarah Chen, a senior P&C claims adjuster with 15 years of experience '
        'in auto and homeowners claims. '
        'You specialize in coverage determination, damage assessment, and fraud detection.\n\n'
        'SCOPE: Auto claims, homeowners claims, coverage questions, and damage assessment.\n\n'
        'GUARDRAILS:\n'
        '- Never provide specific dollar settlement recommendations\n'
        '- Never give legal advice\n'
        '- Always recommend professional inspection for structural damage\n'
        '- Flag potential fraud indicators without making accusations'
    ),
    user_template=(
        'Analyze the following {claim_type} insurance claim.\n\n'
        'Think through this step-by-step:\n'
        '1. What coverage applies?\n'
        '2. What are the key risk factors?\n'
        '3. Are there any red flags?\n'
        '4. What are the recommended next steps?\n\n'
        'Policy limit: ${policy_limit:,}\n\n'
        'Claim: {claim_text}'
    ),
    description='Full persona/scope/guardrails + Chain-of-Thought.'
)

# Variant D: Optimized — concise system prompt + structured output + token-efficient
VARIANT_D = PromptVariant(
    name='D_optimized',
    system_prompt=(
        'Senior P&C claims adjuster. Auto + homeowners specialist.\n'
        'Rules: No dollar settlements. No legal advice. Flag fraud indicators. '
        'Recommend inspections for structural damage.'
    ),
    user_template=(
        'Claim type: {claim_type} | Policy limit: ${policy_limit:,}\n\n'
        '{claim_text}\n\n'
        'Respond in JSON:\n'
        '{{"coverage": "...", "risk_factors": [...], "red_flags": [...], '
        '"next_steps": [...], "priority": "HIGH|MEDIUM|LOW"}}'
    ),
    description='Compressed system prompt + JSON output for token efficiency.'
)

VARIANTS = [VARIANT_A, VARIANT_B, VARIANT_C, VARIANT_D]


# ----- A/B Testing Harness -----
class ABTestHarness:
    """Run multiple prompt variants against the same test cases and compare."""
    
    def __init__(self, variants, test_claims):
        self.variants = variants
        self.test_claims = test_claims
        self.results = []   # list of dicts: {variant, claim_id, response, metrics}
    
    def _render_prompt(self, variant, claim):
        """Fill in the user template with claim data."""
        return variant.user_template.format(
            claim_text=claim['description'],
            claim_type=claim.get('type', 'general'),
            policy_limit=claim.get('policy_limit', 0)
        )
    
    def run(self, verbose=True):
        """Execute all variants against all claims."""
        total = len(self.variants) * len(self.test_claims)
        count = 0
        
        for variant in self.variants:
            if verbose:
                print(f'\n{"="*60}')
                print(f'VARIANT: {variant.name}')
                print(f'  {variant.description}')
                print(f'{"="*60}')
            
            for claim in self.test_claims:
                count += 1
                prompt = self._render_prompt(variant, claim)
                
                if verbose:
                    print(f'\n  [{count}/{total}] {claim["id"]} ({claim["type"]})...', end=' ')
                
                response = ask(prompt, system=variant.system_prompt)
                
                # Quality scoring: quick rule-based checks
                quality = self._score_quality(response['text'], claim)
                
                result = {
                    'variant':       variant.name,
                    'claim_id':      claim['id'],
                    'claim_type':    claim['type'],
                    'response_text': response['text'],
                    'input_tokens':  response['input_tokens'],
                    'output_tokens': response['output_tokens'],
                    'total_tokens':  response['input_tokens'] + response['output_tokens'],
                    'latency':       response['latency'],
                    'quality_score': quality['score'],
                    'quality_checks': quality['checks']
                }
                self.results.append(result)
                
                if verbose:
                    print(f'tokens={result["total_tokens"]}, '
                          f'latency={result["latency"]}s, '
                          f'quality={result["quality_score"]:.0%}')
        
        if verbose:
            print(f'\n{"="*60}')
            print(f'A/B test complete: {total} runs across {len(self.variants)} variants')
    
    def _score_quality(self, text, claim):
        """Rule-based quality scoring (0-1). Checks for expected content."""
        text_lower = text.lower()
        checks = {}
        
        # 1. Mentions coverage determination
        checks['coverage_mentioned'] = any(w in text_lower for w in 
            ['coverage', 'covered', 'policy applies', 'comprehensive', 'collision', 'dwelling'])
        
        # 2. Identifies claim type correctly
        checks['claim_type_correct'] = claim['type'] in text_lower
        
        # 3. Mentions risk factors or red flags
        checks['risk_assessment'] = any(w in text_lower for w in 
            ['risk', 'red flag', 'concern', 'fraud', 'indicator', 'suspicious'])
        
        # 4. Provides next steps
        checks['next_steps'] = any(w in text_lower for w in 
            ['next step', 'recommend', 'should', 'action', 'follow up', 'inspect'])
        
        # 5. Respects guardrails (no specific dollar settlement)
        import re
        dollar_pattern = r'settle for \$[\d,]+|recommend \$[\d,]+|pay \$[\d,]+'
        checks['guardrails_respected'] = not bool(re.search(dollar_pattern, text_lower))
        
        # 6. Substantive response (not too short)
        checks['substantive'] = len(text.split()) > 50
        
        score = sum(checks.values()) / len(checks)
        return {'score': score, 'checks': checks}
    
    def summary(self):
        """Print a comparison table across variants."""
        print(f'\n{"="*80}')
        print('A/B TEST RESULTS SUMMARY')
        print(f'{"="*80}')
        print(f'{"Variant":<20} {"Avg Quality":>12} {"Avg Tokens":>12} '
              f'{"Avg Latency":>12} {"Cost Index":>12}')
        print(f'{"-"*20} {"-"*12} {"-"*12} {"-"*12} {"-"*12}')
        
        variant_stats = {}
        
        for variant in self.variants:
            v_results = [r for r in self.results if r['variant'] == variant.name]
            
            avg_quality = sum(r['quality_score'] for r in v_results) / len(v_results)
            avg_tokens  = sum(r['total_tokens'] for r in v_results) / len(v_results)
            avg_latency = sum(r['latency'] for r in v_results) / len(v_results)
            
            # Cost index: normalized to variant A (baseline = 1.0)
            variant_stats[variant.name] = {
                'quality': avg_quality,
                'tokens':  avg_tokens,
                'latency': avg_latency
            }
        
        # Calculate cost index relative to first variant
        baseline_tokens = variant_stats[self.variants[0].name]['tokens']
        
        for variant in self.variants:
            stats = variant_stats[variant.name]
            cost_idx = stats['tokens'] / baseline_tokens if baseline_tokens > 0 else 0
            
            print(f'{variant.name:<20} {stats["quality"]:>11.0%} '
                  f'{stats["tokens"]:>11.0f} {stats["latency"]:>10.1f}s '
                  f'{cost_idx:>11.2f}x')
        
        # Efficiency score: quality per token (higher = better)
        print(f'\n--- Efficiency Score (Quality / Tokens × 1000) ---')
        best_name, best_score = '', 0
        for variant in self.variants:
            stats = variant_stats[variant.name]
            efficiency = (stats['quality'] / stats['tokens']) * 1000 if stats['tokens'] > 0 else 0
            if efficiency > best_score:
                best_name, best_score = variant.name, efficiency
            print(f'  {variant.name:<20} {efficiency:.3f}')
        
        print(f'\n>>> WINNER: {best_name} (best quality-per-token ratio)')
        return variant_stats


# ----- Run the A/B Test -----
print('Day 28: Prompt Optimization — A/B Testing Framework')
print('=' * 60)
print(f'Model: {model_id}')
print(f'Variants: {len(VARIANTS)}')
print(f'Test claims: {len(TEST_CLAIMS)}')
print(f'Total runs: {len(VARIANTS) * len(TEST_CLAIMS)}')
print()

for v in VARIANTS:
    sys_len = len(v.system_prompt) if v.system_prompt else 0
    print(f'  {v.name:<20} system={sys_len:>4} chars  |  {v.description}')

harness = ABTestHarness(VARIANTS, TEST_CLAIMS)
harness.run(verbose=True)
stats = harness.summary()

Day 28: Prompt Optimization — A/B Testing Framework
Model: us.anthropic.claude-sonnet-4-5-20250929-v1:0
Variants: 4
Test claims: 3
Total runs: 12

  A_minimal            system=   0 chars  |  No system prompt. Minimal user instruction.
  B_structured         system=  84 chars  |  Basic persona + structured output request.
  C_full_cot           system= 496 chars  |  Full persona/scope/guardrails + Chain-of-Thought.
  D_optimized          system= 172 chars  |  Compressed system prompt + JSON output for token efficiency.

VARIANT: A_minimal
  No system prompt. Minimal user instruction.

  [1/12] CLM-001 (auto)... tokens=578, latency=10.41s, quality=83%

  [2/12] CLM-002 (homeowners)... tokens=515, latency=8.26s, quality=67%

  [3/12] CLM-003 (auto)... tokens=466, latency=9.59s, quality=83%

VARIANT: B_structured
  Basic persona + structured output request.

  [4/12] CLM-001 (auto)... tokens=712, latency=12.56s, quality=100%

  [5/12] CLM-002 (homeowners)... tokens=1150, latency=19.45s, q

In [2]:
# =============================================================
# Cell 2 — Token Efficiency & Cost Analysis with LLM-as-Judge
# =============================================================
# Cell 1 showed D_optimized winning on efficiency, but our
# quality scoring was just keyword checks. Now we add LLM-as-judge
# to see if the "winner" holds up under deeper evaluation.
# =============================================================

import re

# ----- LLM-as-Judge Evaluator -----
JUDGE_SYSTEM = """You are an expert insurance claims quality evaluator. 
Score the following claim analysis on these 5 dimensions (each 1-5):

1. ACCURACY: Are coverage determinations and facts correct?
2. REASONING: Is the analysis logically sound with clear justification?
3. COMPLETENESS: Are all key aspects addressed (coverage, risks, next steps)?
4. ACTIONABILITY: Could an adjuster act on this immediately?
5. RISK_AWARENESS: Are red flags and concerns properly identified?

Respond ONLY in this exact format — no other text:
ACCURACY: [1-5]
REASONING: [1-5]
COMPLETENESS: [1-5]
ACTIONABILITY: [1-5]
RISK_AWARENESS: [1-5]"""


def llm_judge_score(response_text, claim):
    """Use LLM-as-judge to evaluate response quality."""
    judge_prompt = f"""Evaluate this claims analysis response.

ORIGINAL CLAIM:
Type: {claim['type']} | Policy limit: ${claim['policy_limit']:,}
{claim['description']}

RESPONSE TO EVALUATE:
{response_text}"""
    
    result = ask(judge_prompt, system=JUDGE_SYSTEM, max_tokens=200, temperature=0.0)
    
    # Parse scores
    scores = {}
    for dimension in ['ACCURACY', 'REASONING', 'COMPLETENESS', 'ACTIONABILITY', 'RISK_AWARENESS']:
        match = re.search(rf'{dimension}:\s*(\d)', result['text'])
        if match:
            scores[dimension] = int(match.group(1))
        else:
            scores[dimension] = 3  # default if parse fails
    
    # Weighted average 
    weights = {
        'ACCURACY': 0.30,
        'REASONING': 0.25,
        'COMPLETENESS': 0.20,
        'ACTIONABILITY': 0.15,
        'RISK_AWARENESS': 0.10
    }
    
    weighted_score = sum(scores[d] * weights[d] for d in scores) / 5  # normalize to 0-1
    
    return {
        'dimensions': scores,
        'weighted_score': weighted_score,
        'judge_tokens': result['input_tokens'] + result['output_tokens']
    }


# ----- Run LLM-as-Judge on all Cell 1 results -----
print("=" * 70)
print("LLM-AS-JUDGE EVALUATION OF A/B TEST RESULTS")
print("=" * 70)
print(f"\nEvaluating {len(harness.results)} responses from Cell 1...\n")

judge_results = {}  # variant_name -> list of judge scores

for r in harness.results:
    variant = r['variant']
    if variant not in judge_results:
        judge_results[variant] = []
    
    # Find the matching claim
    claim = next(c for c in TEST_CLAIMS if c['id'] == r['claim_id'])
    
    print(f"  Judging {variant} / {r['claim_id']}...", end=' ')
    judge = llm_judge_score(r['response_text'], claim)
    judge_results[variant].append({
        'claim_id': r['claim_id'],
        'scores': judge['dimensions'],
        'weighted': judge['weighted_score'],
        'judge_tokens': judge['judge_tokens']
    })
    print(f"weighted={judge['weighted_score']:.2f}  "
          f"({', '.join(f'{d[:3]}={s}' for d, s in judge['dimensions'].items())})")


# ----- Detailed Dimension Breakdown -----
print(f"\n{'=' * 70}")
print("DIMENSION AVERAGES BY VARIANT")
print(f"{'=' * 70}")

dimensions = ['ACCURACY', 'REASONING', 'COMPLETENESS', 'ACTIONABILITY', 'RISK_AWARENESS']
print(f"\n{'Variant':<20}", end='')
for d in dimensions:
    print(f" {d[:6]:>8}", end='')
print(f" {'WEIGHTED':>10}")
print("-" * 80)

variant_judge_avgs = {}

for variant_name in [v.name for v in VARIANTS]:
    results_list = judge_results[variant_name]
    
    dim_avgs = {}
    for d in dimensions:
        dim_avgs[d] = sum(r['scores'][d] for r in results_list) / len(results_list)
    
    avg_weighted = sum(r['weighted'] for r in results_list) / len(results_list)
    variant_judge_avgs[variant_name] = {
        'dimensions': dim_avgs,
        'weighted': avg_weighted
    }
    
    print(f"{variant_name:<20}", end='')
    for d in dimensions:
        print(f" {dim_avgs[d]:>8.1f}", end='')
    print(f" {avg_weighted:>10.3f}")


# ----- Combined Cost Analysis -----
print(f"\n{'=' * 70}")
print("COMBINED COST ANALYSIS: RULE-BASED vs LLM-JUDGE QUALITY")
print(f"{'=' * 70}")

# Claude Sonnet pricing
INPUT_COST_PER_M  = 3.00   # $/million input tokens
OUTPUT_COST_PER_M = 15.00  # $/million output tokens

print(f"\nPricing: ${INPUT_COST_PER_M}/M input tokens, ${OUTPUT_COST_PER_M}/M output tokens")
print(f"Note: Output tokens are {OUTPUT_COST_PER_M/INPUT_COST_PER_M:.0f}x more expensive than input!\n")

print(f"{'Variant':<16} {'Rule Qual':>9} {'Judge Qual':>10} {'Avg In':>8} {'Avg Out':>8} "
      f"{'$/Claim':>9} {'$/10K Claims':>13}")
print("-" * 80)

cost_analysis = {}

for variant in VARIANTS:
    v_results = [r for r in harness.results if r['variant'] == variant.name]
    
    avg_in  = sum(r['input_tokens'] for r in v_results) / len(v_results)
    avg_out = sum(r['output_tokens'] for r in v_results) / len(v_results)
    
    # Cost per claim
    cost_per_claim = (avg_in * INPUT_COST_PER_M + avg_out * OUTPUT_COST_PER_M) / 1_000_000
    cost_10k = cost_per_claim * 10_000
    
    rule_quality = sum(r['quality_score'] for r in v_results) / len(v_results)
    judge_quality = variant_judge_avgs[variant.name]['weighted']
    
    cost_analysis[variant.name] = {
        'avg_in': avg_in,
        'avg_out': avg_out,
        'cost_per_claim': cost_per_claim,
        'cost_10k': cost_10k,
        'rule_quality': rule_quality,
        'judge_quality': judge_quality
    }
    
    print(f"{variant.name:<16} {rule_quality:>8.0%} {judge_quality:>10.3f} "
          f"{avg_in:>8.0f} {avg_out:>8.0f} "
          f"${cost_per_claim:>7.4f} ${cost_10k:>11.2f}")


# ----- Revised Efficiency: Quality per Dollar -----
print(f"\n{'=' * 70}")
print("REVISED EFFICIENCY: Judge Quality per Dollar (higher = better)")
print(f"{'=' * 70}\n")

best_name, best_eff = '', 0
for variant in VARIANTS:
    ca = cost_analysis[variant.name]
    if ca['cost_per_claim'] > 0:
        efficiency = ca['judge_quality'] / ca['cost_per_claim']
    else:
        efficiency = 0
    
    bar = '█' * int(efficiency * 2)
    print(f"  {variant.name:<16} {efficiency:>8.1f}  {bar}")
    
    if efficiency > best_eff:
        best_name, best_eff = variant.name, efficiency

print(f"\n  >>> REVISED WINNER: {best_name}")


# ----- The Key Insight -----
print(f"\n{'=' * 70}")
print("KEY INSIGHT: Rule-Based vs LLM-Judge Rankings")
print(f"{'=' * 70}")

# Sort by each metric
rule_ranked = sorted(cost_analysis.items(), key=lambda x: x[1]['rule_quality'], reverse=True)
judge_ranked = sorted(cost_analysis.items(), key=lambda x: x[1]['judge_quality'], reverse=True)

print(f"\n  {'Rank':<6} {'By Rule-Based':<25} {'By LLM-Judge':<25}")
print(f"  {'-'*6} {'-'*25} {'-'*25}")
for i in range(len(VARIANTS)):
    rn, rv = rule_ranked[i]
    jn, jv = judge_ranked[i]
    print(f"  {i+1:<6} {rn:<18} ({rv['rule_quality']:.0%})    "
          f"{jn:<18} ({jv['judge_quality']:.3f})")

print(f"""
When the rankings differ, it means your fast quality check (rule-based)
is not aligned with your deep quality check (LLM-judge). This is exactly
why the three-tier evaluation pyramid matters — you can't optimize what
you can't accurately measure.
""")

LLM-AS-JUDGE EVALUATION OF A/B TEST RESULTS

Evaluating 12 responses from Cell 1...

  Judging A_minimal / CLM-001... weighted=0.87  (ACC=4, REA=4, COM=5, ACT=5, RIS=4)
  Judging A_minimal / CLM-002... weighted=0.74  (ACC=4, REA=4, COM=3, ACT=4, RIS=3)
  Judging A_minimal / CLM-003... weighted=0.94  (ACC=5, REA=5, COM=4, ACT=5, RIS=4)
  Judging B_structured / CLM-001... weighted=0.87  (ACC=4, REA=4, COM=5, ACT=5, RIS=4)
  Judging B_structured / CLM-002... weighted=0.94  (ACC=5, REA=5, COM=4, ACT=5, RIS=4)
  Judging B_structured / CLM-003... weighted=0.98  (ACC=5, REA=5, COM=5, ACT=5, RIS=4)
  Judging C_full_cot / CLM-001... weighted=0.87  (ACC=4, REA=4, COM=5, ACT=5, RIS=4)
  Judging C_full_cot / CLM-002... weighted=1.00  (ACC=5, REA=5, COM=5, ACT=5, RIS=5)
  Judging C_full_cot / CLM-003... weighted=0.98  (ACC=5, REA=5, COM=5, ACT=5, RIS=4)
  Judging D_optimized / CLM-001... weighted=0.89  (ACC=4, REA=4, COM=5, ACT=5, RIS=5)
  Judging D_optimized / CLM-002... weighted=1.00  (ACC=5, REA

In [3]:
# =============================================================
# Cell 3 — Quality vs Cost: Side-by-Side Deep Dive
# =============================================================
# The numbers say D wins. But numbers can lie. Let's look at the
# actual responses to see if the LLM-judge is being generous
# with JSON format, or if D truly delivers better analysis.
# =============================================================

# ----- Side-by-Side Response Comparison -----
print("=" * 70)
print("SIDE-BY-SIDE RESPONSE COMPARISON")
print("=" * 70)
print("\nFor each claim, we compare the top two variants (B and D)")
print("so you can judge the quality yourself.\n")

for claim in TEST_CLAIMS:
    print(f"\n{'━' * 70}")
    print(f"CLAIM: {claim['id']} ({claim['type']})")
    print(f"{'━' * 70}")
    print(f"\n{claim['description'][:120]}...\n")
    
    # Get B and D responses for this claim
    b_result = next(r for r in harness.results 
                    if r['variant'] == 'B_structured' and r['claim_id'] == claim['id'])
    d_result = next(r for r in harness.results 
                    if r['variant'] == 'D_optimized' and r['claim_id'] == claim['id'])
    
    # Get judge scores
    b_judge = next(j for j in judge_results['B_structured'] 
                   if j['claim_id'] == claim['id'])
    d_judge = next(j for j in judge_results['D_optimized'] 
                   if j['claim_id'] == claim['id'])
    
    print(f"┌─── B_structured (Judge: {b_judge['weighted']:.2f}) "
          f"── {b_result['total_tokens']} tokens ── ${b_result['input_tokens'] * 3 / 1e6 + b_result['output_tokens'] * 15 / 1e6:.4f}")
    print(f"│")
    # Word-wrap the response
    words = b_result['response_text'].split()
    line = "│  "
    for w in words:
        if len(line) + len(w) > 68:
            print(line)
            line = "│  "
        line += w + " "
    if line.strip() != "│":
        print(line)
    
    print(f"│")
    print(f"└─── Dimensions: " + 
          ", ".join(f"{d[:3]}={s}" for d, s in b_judge['scores'].items()))
    
    print()
    
    print(f"┌─── D_optimized (Judge: {d_judge['weighted']:.2f}) "
          f"── {d_result['total_tokens']} tokens ── ${d_result['input_tokens'] * 3 / 1e6 + d_result['output_tokens'] * 15 / 1e6:.4f}")
    print(f"│")
    words = d_result['response_text'].split()
    line = "│  "
    for w in words:
        if len(line) + len(w) > 68:
            print(line)
            line = "│  "
        line += w + " "
    if line.strip() != "│":
        print(line)
    
    print(f"│")
    print(f"└─── Dimensions: " + 
          ", ".join(f"{d[:3]}={s}" for d, s in d_judge['scores'].items()))


# ----- Cost Impact at Scale -----
print(f"\n\n{'=' * 70}")
print("COST IMPACT AT SCALE")
print(f"{'=' * 70}")

volumes = [1000, 5000, 10000, 50000, 100000]

print(f"\n{'Volume':<12}", end='')
for v in VARIANTS:
    print(f" {v.name:>14}", end='')
print()
print("-" * 72)

for vol in volumes:
    print(f"{vol:>10,}  ", end='')
    for variant in VARIANTS:
        ca = cost_analysis[variant.name]
        cost = ca['cost_per_claim'] * vol
        print(f" ${cost:>12,.2f}", end='')
    print()

print(f"\n{'─' * 72}")

# Annual savings: B vs D
b_annual = cost_analysis['B_structured']['cost_per_claim'] * 10000 * 12
d_annual = cost_analysis['D_optimized']['cost_per_claim'] * 10000 * 12
savings = b_annual - d_annual

print(f"\nAnnual projection (10K claims/month):")
print(f"  B_structured: ${b_annual:,.2f}/year")
print(f"  D_optimized:  ${d_annual:,.2f}/year")
print(f"  Savings:      ${savings:,.2f}/year ({savings/b_annual:.0%} reduction)")


# ----- The Real Question -----
print(f"\n\n{'=' * 70}")
print("THE PRODUCTION DECISION FRAMEWORK")
print(f"{'=' * 70}")
print("""
The data gives us three options:

┌─────────────────────────────────────────────────────────────────┐
│  OPTION 1: D_optimized (Best efficiency)                       │
│  Quality: 0.957 | Cost: $0.0062/claim | $738/year @ 10K/mo     │
│  Risk: JSON output may lack nuance for complex claims          │
│  Best for: High-volume automated pipelines                     │
├─────────────────────────────────────────────────────────────────┤
│  OPTION 2: B_structured (Best rule-based quality)              │
│  Quality: 0.930 | Cost: $0.0114/claim | $1,363/year @ 10K/mo   │
│  Risk: Higher cost, more output tokens                         │
│  Best for: Human-reviewed workflows, adjuster-facing output    │
├─────────────────────────────────────────────────────────────────┤
│  OPTION 3: Hybrid (Use case determines variant)                │
│  D for automated triage + B for adjuster review queue          │
│  Route based on claim complexity/severity                      │
│  Best for: Production systems with mixed workflows             │
└─────────────────────────────────────────────────────────────────┘

The right answer depends on WHO reads the output:
  → Machine downstream? Choose D (JSON, cheap, structured)
  → Human adjuster? Choose B (readable, thorough, actionable)
  → Both? Route by use case (Option 3)
""")


# ----- Judge Bias Check -----
print(f"{'=' * 70}")
print("JUDGE BIAS CHECK: Is the LLM generous with JSON?")
print(f"{'=' * 70}")

# Compare average scores for JSON (D) vs prose (B, C) responses
json_avg = variant_judge_avgs['D_optimized']['weighted']
prose_avg = (variant_judge_avgs['B_structured']['weighted'] + 
             variant_judge_avgs['C_full_cot']['weighted']) / 2

print(f"\n  JSON variant (D) avg judge score:   {json_avg:.3f}")
print(f"  Prose variants (B,C) avg judge score: {prose_avg:.3f}")
print(f"  Difference: {json_avg - prose_avg:+.3f}")

if json_avg > prose_avg + 0.02:
    print(f"\n  ⚠ POSSIBLE BIAS: Judge scores JSON higher than prose.")
    print(f"  The structured format may make it easier for the judge to")
    print(f"  'check boxes' without evaluating reasoning depth.")
    print(f"  RECOMMENDATION: Add human evaluation for final validation.")
elif json_avg < prose_avg - 0.02:
    print(f"\n  JSON variant genuinely scores lower — no bias detected.")
else:
    print(f"\n  Scores are comparable — no significant format bias detected.")

SIDE-BY-SIDE RESPONSE COMPARISON

For each claim, we compare the top two variants (B and D)
so you can judge the quality yourself.


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
CLAIM: CLM-001 (auto)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Rear-end collision on Highway 401. Three vehicles involved. Claimant reports whiplash and vehicle damage to rear bumper ...

┌─── B_structured (Judge: 0.87) ── 712 tokens ── $0.0092
│
│  # AUTO INSURANCE CLAIM ANALYSIS ## 1. COVERAGE DETERMINATION 
│  **Primary Coverage Applicable:** - **Collision Coverage**: 
│  Vehicle damage ($8,500) - COVERED - **Bodily Injury/Medical 
│  Payments**: Whiplash injury ($3,200) - COVERED - **Liability 
│  Assessment**: Claimant appears NOT at fault (other driver cited) 
│  **Coverage Position**: **FAVORABLE** - Clear liability against 
│  other party (police citation for following too closely) - 
│  Claimant is middle vehicle in 3-car chain reaction (presumed

In [5]:
# =============================================================
# Cell 4 — Production Optimization Strategy
# =============================================================
# We've measured quality and cost. Now we build the system that
# makes optimization decisions automatically:
# 1. A complexity router that picks the right variant per claim
# 2. A PromptOptimizer class for running experiments
# 3. A deployment recommendation engine
# =============================================================

# ----- Claim Complexity Router -----
# Not all claims need the same prompt investment.
# Simple claims get the cheap variant; complex ones get the thorough one.

def assess_complexity(claim):
    """Score claim complexity 0-1 based on heuristics."""
    score = 0.0
    factors = []
    desc = claim['description'].lower()
    
    # Multiple parties involved
    if any(w in desc for w in ['third party', 'other driver', 'pedestrian', 
                                'cyclist', 'multiple', 'three']):
        score += 0.2
        factors.append('multiple_parties')
    
    # Bodily injury mentioned
    if any(w in desc for w in ['injury', 'pain', 'whiplash', 'medical', 
                                'hospital', 'ambulance']):
        score += 0.2
        factors.append('bodily_injury')
    
    # High dollar amount
    if claim.get('policy_limit', 0) > 100000:
        score += 0.15
        factors.append('high_policy_limit')
    
    # Potential fraud indicators
    if any(w in desc for w in ['no witness', 'left the scene', 'no police', 
                                'overnight', '3 am', '2 am', 'unattended']):
        score += 0.2
        factors.append('fraud_indicators')
    
    # Structural damage (needs inspection)
    if any(w in desc for w in ['structural', 'foundation', 'ceiling', 'roof',
                                'renovation', 'total loss']):
        score += 0.15
        factors.append('structural_damage')
    
    # Subrogation potential
    if any(w in desc for w in ['other driver cited', 'at fault', 'defective',
                                'manufacturer', 'contractor']):
        score += 0.1
        factors.append('subrogation_potential')
    
    return {
        'score': min(score, 1.0),
        'level': 'HIGH' if score >= 0.4 else 'MEDIUM' if score >= 0.2 else 'LOW',
        'factors': factors
    }


def route_to_variant(claim, variants):
    """Pick the best prompt variant based on claim complexity."""
    complexity = assess_complexity(claim)
    
    if complexity['level'] == 'LOW':
        # Simple claims: cheap and fast
        pick = next(v for v in variants if v.name == 'D_optimized')
        reason = "Low complexity — use token-efficient variant"
    elif complexity['level'] == 'MEDIUM':
        # Medium: structured prose for adjuster readability
        pick = next(v for v in variants if v.name == 'B_structured')
        reason = "Medium complexity — use structured prose for adjuster review"
    else:
        # High: full CoT for thorough analysis
        pick = next(v for v in variants if v.name == 'C_full_cot')
        reason = "High complexity — use full Chain-of-Thought analysis"
    
    return {
        'variant': pick,
        'complexity': complexity,
        'reason': reason
    }


# ----- Test the Router -----
print("=" * 70)
print("COMPLEXITY-BASED ROUTING")
print("=" * 70)

# Original test claims
print("\n--- Original Test Claims ---\n")
for claim in TEST_CLAIMS:
    routing = route_to_variant(claim, VARIANTS)
    cx = routing['complexity']
    print(f"  {claim['id']} ({claim['type']:<12}) → "
          f"Complexity: {cx['level']:<6} ({cx['score']:.2f}) → "
          f"{routing['variant'].name}")
    if cx['factors']:
        print(f"    Factors: {', '.join(cx['factors'])}")

# Additional test claims to stress-test the router
ROUTING_TEST_CLAIMS = [
    {
        'id': 'CLM-100',
        'type': 'auto',
        'description': 'Minor fender bender in parking lot. Small dent on rear bumper. '
                       'No injuries. Both drivers exchanged info. Estimated repair: $1,200.',
        'policy_limit': 25000
    },
    {
        'id': 'CLM-101',
        'type': 'homeowners',
        'description': 'Policyholder reports roof damage from last night\'s hailstorm. '
                       'Multiple shingles cracked. No interior damage visible. '
                       'Neighbor also filing claims. Estimated repair: $4,500.',
        'policy_limit': 250000
    },
    {
        'id': 'CLM-102',
        'type': 'auto',
        'description': 'Multi-vehicle accident on Highway 401. Insured rear-ended by '
                       'third party who left the scene. Insured then pushed into vehicle ahead. '
                       'Insured reports neck pain and headaches. Passenger taken by ambulance. '
                       'Police report filed. Other driver cited at fault. Estimated vehicle '
                       'damage: $22,000. Medical bills: $15,000 and climbing. '
                       'Vehicle may be total loss.',
        'policy_limit': 500000
    },
    {
        'id': 'CLM-103',
        'type': 'homeowners',
        'description': 'Insured reports jewelry stolen from home. Claims a $35,000 watch '
                       'and $12,000 necklace were taken. Discovered missing after returning '
                       'from two-week vacation. No signs of forced entry. No police report '
                       'filed. Alarm system was not activated. No witnesses.',
        'policy_limit': 400000
    }
]

print("\n\n--- Additional Routing Tests ---\n")
for claim in ROUTING_TEST_CLAIMS:
    routing = route_to_variant(claim, VARIANTS)
    cx = routing['complexity']
    print(f"  {claim['id']} ({claim['type']:<12}) → "
          f"Complexity: {cx['level']:<6} ({cx['score']:.2f}) → "
          f"{routing['variant'].name}")
    if cx['factors']:
        print(f"    Factors: {', '.join(cx['factors'])}")
    print(f"    Reason: {routing['reason']}")


# ----- PromptOptimizer: The Production Class -----
class PromptOptimizer:
    """Production prompt optimization engine.
    
    Combines A/B testing, complexity routing, and cost tracking
    into a single interface for managing prompt performance.
    """
    
    def __init__(self, variants, quality_threshold=0.85):
        self.variants = {v.name: v for v in variants}
        self.quality_threshold = quality_threshold
        self.experiment_log = []
    
    def analyze_claim(self, claim, variant_name=None):
        """Process a claim with routing or a specified variant."""
        if variant_name:
            variant = self.variants[variant_name]
            routing_reason = f"Manual override: {variant_name}"
        else:
            routing = route_to_variant(claim, list(self.variants.values()))
            variant = routing['variant']
            routing_reason = routing['reason']
        
        # Render and call
        prompt = variant.user_template.format(
            claim_text=claim['description'],
            claim_type=claim.get('type', 'general'),
            policy_limit=claim.get('policy_limit', 0)
        )
        
        response = ask(prompt, system=variant.system_prompt)
        
        # Log
        entry = {
            'claim_id':     claim['id'],
            'variant':      variant.name,
            'routing':      routing_reason,
            'input_tokens': response['input_tokens'],
            'output_tokens': response['output_tokens'],
            'total_tokens': response['input_tokens'] + response['output_tokens'],
            'latency':      response['latency'],
            'cost':         (response['input_tokens'] * INPUT_COST_PER_M + 
                           response['output_tokens'] * OUTPUT_COST_PER_M) / 1_000_000,
            'response':     response['text']
        }
        self.experiment_log.append(entry)
        
        return entry
    
    def run_experiment(self, claims, verbose=True):
        """Run routed analysis on a batch of claims."""
        if verbose:
            print(f"\nProcessing {len(claims)} claims with complexity routing...\n")
        
        for claim in claims:
            entry = self.analyze_claim(claim)
            if verbose:
                print(f"  {entry['claim_id']:<10} → {entry['variant']:<16} "
                      f"tokens={entry['total_tokens']:<6} "
                      f"latency={entry['latency']}s  "
                      f"cost=${entry['cost']:.4f}")
                print(f"    Routing: {entry['routing']}")
        
        return self.experiment_log
    
    def deployment_report(self):
        """Generate a deployment recommendation report."""
        if not self.experiment_log:
            print("No experiments run yet.")
            return
        
        print(f"\n{'=' * 70}")
        print("DEPLOYMENT RECOMMENDATION REPORT")
        print(f"{'=' * 70}")
        print(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
        print(f"Claims analyzed: {len(self.experiment_log)}")
        print(f"Quality threshold: {self.quality_threshold:.0%}\n")
        
        # Routing distribution
        variant_counts = {}
        variant_costs = {}
        variant_tokens = {}
        
        for entry in self.experiment_log:
            v = entry['variant']
            variant_counts[v] = variant_counts.get(v, 0) + 1
            variant_costs[v] = variant_costs.get(v, 0) + entry['cost']
            variant_tokens[v] = variant_tokens.get(v, 0) + entry['total_tokens']
        
        total_claims = len(self.experiment_log)
        total_cost = sum(e['cost'] for e in self.experiment_log)
        total_tokens = sum(e['total_tokens'] for e in self.experiment_log)
        avg_latency = sum(e['latency'] for e in self.experiment_log) / total_claims
        
        print("ROUTING DISTRIBUTION:")
        for v, count in sorted(variant_counts.items()):
            pct = count / total_claims * 100
            bar = '█' * int(pct / 2)
            print(f"  {v:<16} {count:>3} ({pct:>5.1f}%)  {bar}")
        
        print(f"\nCOST SUMMARY:")
        print(f"  Total cost:         ${total_cost:.4f}")
        print(f"  Avg cost/claim:     ${total_cost/total_claims:.4f}")
        print(f"  Total tokens:       {total_tokens:,}")
        print(f"  Avg tokens/claim:   {total_tokens/total_claims:,.0f}")
        print(f"  Avg latency:        {avg_latency:.1f}s")
        
        # Compare: routed vs single-variant
        print(f"\nCOST COMPARISON: Routing vs Single-Variant")
        print(f"  {'Strategy':<25} {'Est. $/10K Claims':>18}")
        print(f"  {'-'*25} {'-'*18}")
        
        routed_10k = (total_cost / total_claims) * 10000
        print(f"  {'Complexity routing':<25} ${routed_10k:>16,.2f}")
        
        for variant_name, ca in cost_analysis.items():
            cost_10k = ca['cost_per_claim'] * 10000
            savings_pct = (cost_10k - routed_10k) / cost_10k * 100 if cost_10k > 0 else 0
            indicator = f"({savings_pct:+.0f}%)" if savings_pct != 0 else ""
            print(f"  {variant_name:<25} ${cost_10k:>16,.2f} {indicator}")
        
        # Recommendation
        print(f"\n{'─' * 70}")
        print("RECOMMENDATION:")
        print(f"{'─' * 70}")
        print(f"""
  Deploy complexity-based routing with three tiers:

  ┌─────────────┬──────────────────┬────────────────────────────────────┐
  │ Complexity  │ Variant          │ Use Case                           │
  ├─────────────┼──────────────────┼────────────────────────────────────┤
  │ LOW         │ D_optimized      │ Automated triage, API responses    │
  │ MEDIUM      │ B_structured     │ Adjuster queue, human review       │
  │ HIGH        │ C_full_cot       │ Complex claims, litigation risk    │
  └─────────────┴──────────────────┴────────────────────────────────────┘

  This approach optimizes cost for simple claims while preserving
  quality for complex ones. Monitor routing distribution monthly —
  if HIGH claims exceed 30%, review complexity thresholds.

  NEXT STEPS:
  1. Run human evaluation on 50+ routed claims to validate quality
  2. Add routing metrics to CloudWatch dashboard
  3. Set up A/B test: routed vs C_full_cot for 2 weeks
  4. Review monthly: cost trends, quality scores, routing distribution
""")


# ----- Run the Optimizer -----
print("=" * 70)
print("PRODUCTION PROMPT OPTIMIZER")
print("=" * 70)

all_claims = TEST_CLAIMS + ROUTING_TEST_CLAIMS
optimizer = PromptOptimizer(VARIANTS, quality_threshold=0.85)
optimizer.run_experiment(all_claims, verbose=True)
optimizer.deployment_report()

COMPLEXITY-BASED ROUTING

--- Original Test Claims ---

  CLM-001 (auto        ) → Complexity: HIGH   (0.50) → C_full_cot
    Factors: multiple_parties, bodily_injury, subrogation_potential
  CLM-002 (homeowners  ) → Complexity: HIGH   (0.50) → C_full_cot
    Factors: high_policy_limit, fraud_indicators, structural_damage
  CLM-003 (auto        ) → Complexity: LOW    (0.00) → D_optimized


--- Additional Routing Tests ---

  CLM-100 (auto        ) → Complexity: LOW    (0.00) → D_optimized
    Reason: Low complexity — use token-efficient variant
  CLM-101 (homeowners  ) → Complexity: HIGH   (0.50) → C_full_cot
    Factors: multiple_parties, high_policy_limit, structural_damage
    Reason: High complexity — use full Chain-of-Thought analysis
  CLM-102 (auto        ) → Complexity: HIGH   (1.00) → C_full_cot
    Factors: multiple_parties, bodily_injury, high_policy_limit, fraud_indicators, structural_damage, subrogation_potential
    Reason: High complexity — use full Chain-of-Thought anal